# Dự đoán Ảnh (Inference)\nNotebook này giúp bạn tải mô hình đã huấn luyện (`best_model.pth`) để phân loại một bức ảnh bất kỳ xem đó là Xe Đạp (Bike) hay Xe Máy (Motorbike).

In [ ]:
import os\nimport torch\nimport torch.nn as nn\nfrom torchvision import transforms\nimport torchvision.models as tv_models\nfrom PIL import Image\nimport matplotlib.pyplot as plt\n\n# Thiết lập thư mục và device\nPROJECT_DIR = os.path.abspath(os.getcwd())\nMODEL_PATH = os.path.join(PROJECT_DIR, 'best_model.pth')\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Đang sử dụng thiết bị: {device}')

### 1. Hàm tải mô hình

In [ ]:
def load_model():\n    if not os.path.exists(MODEL_PATH):\n        print(f'LỖI: Không tìm thấy file {MODEL_PATH}. Vui lòng chạy train.ipynb trước!')\n        return None\n        \n    # Khởi tạo lại kiến trúc giống lúc train\n    model = tv_models.resnet50()\n    num_ftrs = model.fc.in_features\n    model.fc = nn.Linear(num_ftrs, 2)\n    \n    # Load trọng số\n    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))\n    model.to(device)\n    model.eval()\n    return model\n\nmodel = load_model()\nif model:\n    print('Đã tải mô hình thành công!')

### 2. Hàm xử lý ảnh và dự đoán

In [ ]:
def predict_image(image_path, model):\n    class_names = ['Bike', 'Motorbike']\n    \n    # Transform chuẩn của tập validation\n    transform = transforms.Compose([\n        transforms.Resize((224, 224)),\n        transforms.ToTensor(),\n        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])\n    ])\n    \n    try:\n        image = Image.open(image_path).convert('RGB')\n    except Exception as e:\n        print(f'Lỗi khi đọc ảnh: {e}')\n        return\n\n    input_tensor = transform(image).unsqueeze(0).to(device)\n    \n    with torch.no_grad():\n        outputs = model(input_tensor)\n        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)\n        _, preds = torch.max(outputs, 1)\n        \n        pred_class = class_names[preds[0]]\n        confidence = probabilities[preds[0]].item() * 100\n        \n    # Vẽ ảnh và in kết quả\n    plt.figure(figsize=(6, 6))\n    plt.imshow(image)\n    plt.axis('off')\n    plt.title(f'Dự đoán: {pred_class} ({confidence:.2f}%)')\n    plt.show()\n    \n    print(f'Chi tiết độ tự tin: Bike: {probabilities[0]*100:.2f}% | Motorbike: {probabilities[1]*100:.2f}%')

### 3. Chạy thử nghiệm\nBạn hãy thay đổi `image_path` bên dưới thành đường dẫn tới bức ảnh bạn muốn thử.

In [ ]:
if model:\n    # THAY ĐƯỜNG DẪN ẢNH CỦA BẠN VÀO ĐÂY\n    test_image_path = 'duong_dan_toi_anh_cua_ban.jpg'\n    \n    if os.path.exists(test_image_path):\n        predict_image(test_image_path, model)\n    else:\n        print(f'Vui lòng tải 1 bức ảnh lên và thay đường dẫn {test_image_path} bằng đường dẫn thật!')